# Demo 08: Agent Evaluation**Story**: "How good is your agent?"Difficulty scaling, multi-agent comparison, and leaderboards.

In [ ]:
import sys, os, mathsys.path.insert(0, os.path.join(os.getcwd(), "demos"))from _shared import make_disease_system, oracle_agent, random_agent, zero_agentfrom alienbio.bio import AgentInterface, DiagnoseTaskfrom alienbio.scenarios.difficulty_curve import DifficultySpec, measure_difficulty_curvefrom alienbio.bio.comparison import AgentStats, ComparisonTablefrom alienbio.viz import difficulty_curve_plot, agent_comparison_chart

## Setup Difficulty Levels

In [ ]:
spec = DifficultySpec(levels=[])for level, label, n_cand in [(1, "easy", 2), (2, "medium", 4), (3, "hard", 8)]:    system, baseline, perturbs = make_disease_system(seed=level*10)    tasks = []    for i in range(min(3, len(perturbs))):        candidates = perturbs[:n_cand] if n_cand <= len(perturbs) else perturbs        tasks.append(DiagnoseTask(candidates, applied_index=i % len(candidates)))    spec.add_level(level, label, tasks)

## Measure Agent Performance

In [ ]:
system, _, _ = make_disease_system(seed=42)iface = AgentInterface(system)curves = []for name, fn in [("oracle", oracle_agent), ("random", random_agent), ("zero", zero_agent)]:    curve = measure_difficulty_curve(spec, iface, fn, agent_name=name)    curves.append(curve)    for pt in curve.points:        print(f"{name} @ {pt.label}: score={pt.mean_score:.2f}")

In [ ]:
difficulty_curve_plot(curves, title="Difficulty Curves")

## Agent Comparison

In [ ]:
all_stats = []for curve in curves:    scores = [s for pt in curve.points for s in pt.scores]    n = len(scores) if scores else 1    mean = sum(scores)/n if scores else 0.0    var = sum((s-mean)**2 for s in scores)/n if scores else 0.0    pass_rate = sum(1 for s in scores if s >= 0.5)/n if scores else 0.0    all_stats.append(AgentStats(curve.agent_name, mean, math.sqrt(var),        min(scores) if scores else 0.0, max(scores) if scores else 0.0, n, pass_rate))table = ComparisonTable(agents=all_stats)for i, a in enumerate(table.ranking):    print(f"#{i+1} {a.agent_name}: {a.mean:.2f} +/- {a.std:.2f}")

In [ ]:
agent_comparison_chart(table, title="Agent Comparison")

**Takeaway**: The framework systematically evaluates and compares agent capabilities.